In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv("raw_data.csv")

# Print first five rows
print(df.head(5))

null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / df.shape[0]) * 100
print(pd.DataFrame({'Count': null_counts, 'Percentage': null_pct}))

# Check and remove duplicates
print("Duplicate count:", df.duplicated().sum())
df = df.drop_duplicates()
print("Post-removal shape:", df.shape)

# Memory footprint baseline
mem_before = df.memory_usage(deep=True).sum()

# 1. Clean and convert incorrectly inferred object column to numeric
df['Revenue'] = df['Revenue'].astype(str).str.replace('$', '', regex=False)
df['Revenue'] = pd.to_numeric(df['Revenue'], errors='coerce') 

# 2. Convert repetitive low-cardinality string column to categorical
df['Region'] = df['Region'].astype('category')

# Memory footprint evaluation
mem_after = df.memory_usage(deep=True).sum()

# Descriptive statistics summary
print(df[['Revenue', 'Customer_Tenure', 'Satisfaction_Score']].describe())

# Compute Skewness
for col in ['Revenue', 'Customer_Tenure', 'Satisfaction_Score']:
    print(f"{col} Skewness: {df[col].skew():.4f}")

for col in ['Customer_Tenure', 'Satisfaction_Score']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"{col} Outliers: {outliers.shape[0]} rows outside [{lower_bound:.2f}, {upper_bound:.2f}]")


# 1. Line Plot: Ordered Time Trend
plt.figure(figsize=(6, 3))
plt.plot(df.index, df['Revenue'], color='royalblue', alpha=0.6)
plt.title('Revenue Sequential Flow Profile')
plt.xlabel('Row Sequence Index')
plt.ylabel('Revenue Value ($)')
plt.tight_layout(); plt.show()

# 2. Bar Chart: Categorical Group Comparisons
plt.figure(figsize=(6, 3))
df.groupby('Region')['Revenue'].mean().plot(kind='bar', color='teal')
plt.title('Mean Revenue Contribution by Geographic Region')
plt.xlabel('Geographic Region')
plt.ylabel('Mean Revenue ($)')
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

# 3. Histogram: Highly Skewed Distribution Shape
plt.figure(figsize=(6, 3))
sns.histplot(df['Customer_Tenure'], bins=20, kde=True, color='crimson')
plt.title('Distribution of Customer Tenure (Highly Skewed)')
plt.xlabel('Tenure Duration')
plt.ylabel('Frequency Count')
plt.tight_layout(); plt.show()


# 4. Scatter Plot: Variable Correlation Verification
plt.figure(figsize=(6, 3))
sns.scatterplot(data=df, x='Customer_Tenure', y='Satisfaction_Score', alpha=0.5, color='purple')
plt.title('Customer Tenure vs Satisfaction Score Relationship')
plt.xlabel('Customer Tenure')
plt.ylabel('Satisfaction Score')
plt.tight_layout(); plt.show()

# 5. Box Plot: Split Variance Analysis
plt.figure(figsize=(6, 3))
sns.boxplot(data=df, x='Region', y='Revenue', palette='Set2')
plt.title('Revenue Variance Distribution Across Regions')
plt.xlabel('Region')
plt.ylabel('Revenue ($)')
plt.tight_layout(); plt.show()


# Heatmap Visualization
plt.figure(figsize=(6, 4))
sns.heatmap(df[['Revenue', 'Customer_Tenure', 'Satisfaction_Score']].corr(), annot=True, cmap='coolwarm', fmt=".4f")
plt.title('Pearson Linear Correlation Heatmap')
plt.tight_layout(); plt.show()

# Execute targeted robust median imputation
for col in ['Customer_Tenure', 'Satisfaction_Score', 'Revenue']:
    col_median = df[col].median()
    df[col] = df[col].fillna(col_median)

# Sanity verify completion
print("Remaining Nulls:\n", df[['Customer_Tenure', 'Satisfaction_Score', 'Revenue']].isnull().sum())

# Compute matrices
pearson_m = df[['Revenue', 'Customer_Tenure', 'Satisfaction_Score']].corr(method='pearson')
spearman_m = df[['Revenue', 'Customer_Tenure', 'Satisfaction_Score']].corr(method='spearman')
diff_m = (spearman_m - pearson_m).abs()

# Metrics broken down across categories
grouped_stats = df.groupby('Region', observed=False)['Revenue'].agg(['mean', 'std', 'count'])
print(grouped_stats)

# Export clean dataset to CSV
df.to_csv('cleaned_data.csv', index=False)


